# GD lab results download

**Running sequence (top to bottom):**

1. **Cell 1** — install packages (first time only).
2. **Cell 2** — load credentials from `.env`.
3. **Cell 3** — fetch API, compare file hash to existing CSV; save only if changed.

**Outputs** (`downloads/`):

- `gd_labresults.csv` — upload this file to the Databricks volume for `03_clean_gd`
- `gd_labresults_raw.json` — full API response

Upload path: `/Volumes/decide_catalog/decide_schema/decide_volume/gd_labresults.csv`

In [ ]:
%pip install -q requests pandas python-dotenv

In [7]:
import os
from pathlib import Path

from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
load_dotenv(NOTEBOOK_DIR / ".env")

CLIENT_ID = os.environ["GD_CLIENT_ID"]
CLIENT_SECRET = os.environ["GD_CLIENT_SECRET"]
TOKEN_URL = os.getenv(
    "GD_TOKEN_URL",
    "https://services-acc.gdanimalhealth.com/oauth2/resources/tokenservice",
)
RESULTS_URL = os.getenv(
    "GD_RESULTS_URL",
    "https://services-acc.gdanimalhealth.com/api/Decide/1.0/labresults",
)
OUTPUT_DIR = NOTEBOOK_DIR / "downloads"

print("Credentials loaded from .env")

Credentials loaded from .env


In [8]:
import hashlib
import json

import pandas as pd
import requests
from IPython.display import display


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(65536), b""):
            digest.update(chunk)
    return digest.hexdigest()


token_response = requests.post(
    TOKEN_URL,
    data={"grant_type": "client_credentials"},
    auth=(CLIENT_ID, CLIENT_SECRET),
    timeout=60,
)
token_response.raise_for_status()
access_token = token_response.json()["access_token"]

results_response = requests.get(
    RESULTS_URL,
    headers={"Authorization": f"Bearer {access_token}"},
    timeout=120,
)
results_response.raise_for_status()
payload = results_response.json()

if payload.get("serviceStatus", {}).get("result") != "OK":
    raise RuntimeError(payload.get("serviceStatus"))

rows = payload.get("labresults") or []
df = pd.DataFrame(rows)
new_csv_bytes = df.to_csv(index=False).encode("utf-8")
new_csv_hash = sha256_bytes(new_csv_bytes)
new_json_bytes = json.dumps(payload, indent=2, ensure_ascii=False).encode("utf-8")
new_json_hash = sha256_bytes(new_json_bytes)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUTPUT_DIR / "gd_labresults.csv"
json_path = OUTPUT_DIR / "gd_labresults_raw.json"

previous_csv_hash = sha256_file(csv_path) if csv_path.is_file() else None
previous_json_hash = sha256_file(json_path) if json_path.is_file() else None

if previous_csv_hash == new_csv_hash:
    print("File hash is the same as the existing gd_labresults.csv; no need to download again.")
    print(f"SHA-256: {new_csv_hash}")
else:
    csv_path.write_bytes(new_csv_bytes)
    json_path.write_bytes(new_json_bytes)
    if previous_csv_hash is None:
        print("First download saved.")
    else:
        print("Newer version downloaded and saved (CSV content changed).")
    print(f"Rows: {len(df)}")
    print(f"CSV hash:  {new_csv_hash}")
    print(f"JSON hash: {new_json_hash}")
    if previous_csv_hash:
        print(f"Previous CSV hash: {previous_csv_hash}")
    print(f"CSV:  {csv_path}")
    print(f"JSON: {json_path}")
    display(df)

First download saved.
Rows: 5
CSV hash:  5df9d3712138090cf91de56efd630363097afc6c2b9935d657b54e77c1c5eb0d
JSON hash: 096753f22a988201f7e5f002456ec410eccbd1e17b3e7c429b8592ddb9adf31e
CSV:  c:\Users\pn287\OneDrive - Cornell University\Post doc\Post doc Cornell\Project\DECIDE\cattle-use-case-barometer-dev\GD API data query\downloads\gd_labresults.csv
JSON: c:\Users\pn287\OneDrive - Cornell University\Post doc\Post doc Cornell\Project\DECIDE\cattle-use-case-barometer-dev\GD API data query\downloads\gd_labresults_raw.json


,labReference,country,breed,testDate,province,farmID,diagnosticTest,sampleType,pathogen,result
0,2,The Netherlands,Dairy,2024-07-01T00:00:00.000+02:00,Friesland,05758cd3875ad2171484c0026ccbb8adc210cd2d852407...,PCR,Autopsy,MB,0
1,2,The Netherlands,Dairy,2024-08-08T00:00:00.000+02:00,Overijssel,05758cd3875ad2171484c0026ccbb8adc210cd2d852407...,PCR,Autopsy,BRSV,1
2,2,The Netherlands,Dairy,2024-09-13T00:00:00.000+02:00,Drenthe,05758cd3875ad2171484c0026ccbb8adc210cd2d852407...,PCR,BAL,PM,2
3,2,The Netherlands,Dairy,2024-10-19T00:00:00.000+02:00,Limburg,05758cd3875ad2171484c0026ccbb8adc210cd2d852407...,PCR,Autopsy,PM,3
4,2,The Netherlands,Dairy,2024-11-26T00:00:00.000+01:00,Zeeland,05758cd3875ad2171484c0026ccbb8adc210cd2d852407...,Culture,BAL,BRSV,4
